# Pedalia · 3.2.1 · La tarifa, en una función

Solución de referencia: el notebook que construye la práctica de la guía, ejecutado. Sirve para comparar con el nuestro.

[← Volver a la guía de la sesión](3.2.1%20-%20Funciones%2C%20par%C3%A1metros%2C%20valores%20por%20defecto%20y%20manejo%20de%20errores.md)

## La tarifa como función

In [1]:
def bloques_extra(minutos):
    """Bloques de 30 minutos (o fracción) que pasan de los 30 primeros."""
    exceso = minutos - 30
    if exceso <= 0:
        return 0
    elif exceso % 30 == 0:
        return exceso // 30
    else:
        return exceso // 30 + 1


def precio_viaje(minutos, usuario, bici="mecánica", suplemento=0.40):
    """Precio de un viaje de Pedalia, en euros, según la tarifa vigente.

    minutos     duración del viaje, en minutos (mayor que cero)
    usuario     "abonado" u "ocasional"
    bici        "mecánica" (por defecto) o "eléctrica"
    suplemento  suplemento por bici eléctrica, en euros (0,40 por defecto)
    """
    if minutos <= 0:
        raise ValueError(f"duración no válida: {minutos}")

    if usuario == "abonado":
        precio = bloques_extra(minutos) * 0.50
    elif usuario == "ocasional":
        precio = 1.50 + bloques_extra(minutos) * 1.00
    else:
        raise ValueError(f"tipo de usuario desconocido: {usuario!r}")

    if bici == "eléctrica":
        precio += suplemento
    return round(precio, 2)

## Probar la función

In [2]:
assert bloques_extra(30) == 0
assert bloques_extra(31) == 1
assert bloques_extra(60) == 1
assert bloques_extra(61) == 2
assert precio_viaje(30, "abonado") == 0
assert precio_viaje(31, "abonado") == 0.50
assert precio_viaje(64, "ocasional", "eléctrica") == 3.90
assert precio_viaje(22, "ocasional", bici="eléctrica", suplemento=0.60) == 2.10
print("Todas las pruebas pasan")

Todas las pruebas pasan


In [3]:
try:
    precio_viaje(25, "Abonado")
except ValueError as error:
    print("ValueError:", error)

ValueError: tipo de usuario desconocido: 'Abonado'


## Los viajes de la mañana

In [4]:
viajes = [
    {"id": 1,  "origen": "Plaza Mayor",          "destino": "Campus Universitario", "minutos": 14, "usuario": "abonado",   "bici": "mecánica"},
    {"id": 2,  "origen": "Estación Central",     "destino": "Plaza Mayor",          "minutos": 9,  "usuario": "abonado",   "bici": "eléctrica"},
    {"id": 3,  "origen": "Campus Universitario", "destino": "Parque del Río",       "minutos": 38, "usuario": "ocasional", "bici": "mecánica"},
    {"id": 4,  "origen": "Parque del Río",       "destino": "Mercado Central",      "minutos": 22, "usuario": "ocasional", "bici": "eléctrica"},
    {"id": 5,  "origen": "Hospital General",     "destino": "Estación Central",     "minutos": 31, "usuario": "abonado",   "bici": "mecánica"},
    {"id": 6,  "origen": "Mercado Central",      "destino": "Plaza Mayor",          "minutos": 6,  "usuario": "abonado",   "bici": "mecánica"},
    {"id": 7,  "origen": "Plaza Mayor",          "destino": "Parque del Río",       "minutos": 75, "usuario": "ocasional", "bici": "mecánica"},
    {"id": 8,  "origen": "Estación Central",     "destino": "Hospital General",     "minutos": 17, "usuario": "abonado",   "bici": "eléctrica"},
    {"id": 9,  "origen": "Campus Universitario", "destino": "Plaza Mayor",          "minutos": 12, "usuario": "abonado",   "bici": "mecánica"},
    {"id": 10, "origen": "Plaza Mayor",          "destino": "Estación Central",     "minutos": 11, "usuario": "abonado",   "bici": "eléctrica"},
    {"id": 11, "origen": "Parque del Río",       "destino": "Parque del Río",       "minutos": 64, "usuario": "ocasional", "bici": "eléctrica"},
    {"id": 12, "origen": "Mercado Central",      "destino": "Campus Universitario", "minutos": 19, "usuario": "abonado",   "bici": "mecánica"},
]


def ingresos_por_tipo(viajes, suplemento=0.40):
    """Ingresos por viaje de cada tipo de usuario, en euros: {usuario: ingresos}."""
    ingresos = {}
    for viaje in viajes:
        precio = precio_viaje(viaje["minutos"], viaje["usuario"], viaje["bici"], suplemento)
        ingresos[viaje["usuario"]] = ingresos.get(viaje["usuario"], 0) + precio
    return {usuario: round(total, 2) for usuario, total in ingresos.items()}


ingresos_por_tipo(viajes)

{'abonado': 1.7, 'ocasional': 11.8}

## El registro de la tarde

In [5]:
registro_tarde = [
    "13;Plaza Mayor;Mercado Central;8;abonado;mecánica",
    "14;Campus Universitario;Plaza Mayor;26;abonado;eléctrica",
    "15;Parque del Río;Parque del Río;95;ocasional;mecánica",
    "16;Estación Central;Hospital General;doce;abonado;mecánica",
    "17;Mercado Central;Parque del Río;41;Ocasional ;eléctrica",
    "18;Hospital General;Estación Central;-3;abonado;mecánica",
    "19;Plaza Mayor;Campus Universitario;33;abonado;eléctrica",
    "20;Parque del Río;Plaza Mayor;57;turista;mecánica",
    "21;Campus Universitario;Mercado Central;15",
]

In [6]:
def leer_viaje(linea):
    """Convierte una línea del registro ("id;origen;destino;minutos;usuario;bici")
    en un diccionario de viaje con su precio. Lanza ValueError si la línea no es válida."""
    id_viaje, origen, destino, minutos, usuario, bici = linea.split(";")
    viaje = {
        "id": int(id_viaje),
        "origen": origen.strip(),
        "destino": destino.strip(),
        "minutos": int(minutos),
        "usuario": usuario.strip().lower(),
        "bici": bici.strip().lower(),
    }
    viaje["precio"] = precio_viaje(viaje["minutos"], viaje["usuario"], viaje["bici"])
    return viaje

In [7]:
validos = []
rechazados = []
for linea in registro_tarde:
    try:
        validos.append(leer_viaje(linea))
    except ValueError as error:
        rechazados.append((linea, str(error)))

print(f"Válidos: {len(validos)} · rechazados: {len(rechazados)}")
for viaje in validos:
    print(f"  {viaje['id']}  {viaje['usuario']:<10}{viaje['bici']:<10}"
          f"{viaje['minutos']:>3} min {viaje['precio']:>5.2f} €")
for linea, motivo in rechazados:
    print(f"  Viaje {linea.split(';')[0]} rechazado: {motivo}")

Válidos: 5 · rechazados: 4
  13  abonado   mecánica    8 min  0.00 €
  14  abonado   eléctrica  26 min  0.40 €
  15  ocasional mecánica   95 min  4.50 €
  17  ocasional eléctrica  41 min  2.90 €
  19  abonado   eléctrica  33 min  0.90 €
  Viaje 16 rechazado: invalid literal for int() with base 10: 'doce'
  Viaje 18 rechazado: duración no válida: -3
  Viaje 20 rechazado: tipo de usuario desconocido: 'turista'
  Viaje 21 rechazado: not enough values to unpack (expected 6, got 4)


## Escenarios del suplemento eléctrico

In [8]:
jornada = viajes + validos

for suplemento in [0.40, 0.60, 0.80]:
    ingresos = ingresos_por_tipo(jornada, suplemento)
    print(f"Suplemento {suplemento:.2f} €: {sum(ingresos.values()):6.2f} € · {ingresos}")

Suplemento 0.40 €:  22.20 € · {'abonado': 3.0, 'ocasional': 19.2}
Suplemento 0.60 €:  23.80 € · {'abonado': 4.0, 'ocasional': 19.8}
Suplemento 0.80 €:  25.40 € · {'abonado': 5.0, 'ocasional': 20.4}


## La tarifa, en una función

- La tarifa vive ahora en `precio_viaje`, probada con los casos límite. Da las
  mismas cifras que el cálculo de la 3.1.3: 13,50 € de ingresos por viaje en la
  mañana del martes.
- De los nueve viajes de la tarde, cuatro llegan defectuosos (una duración en
  letra, una negativa, un tipo de usuario desconocido y una línea incompleta).
  Hay que avisar al equipo de la aplicación.
- Con los 17 viajes válidos del martes, cada 0,20 € de suplemento eléctrico
  añadirían 1,60 € al día, 1 € de ellos pagado por abonados. Antes de proponer
  una subida habrá que ver si reduce el uso de la eléctrica (parte 3.3).